  ÉTAPE 3 — SYSTÈME DE RECOMMANDATION      

In [20]:
import numpy as np
import joblib # Charger un objet Utilisation principale	Sauvegarde et chargement des modèles ML

3.1  Charger le modèle et le scaler ────────────────────────────────────

In [21]:
print("=" * 150)
print("  3.1  CHARGEMENT DU MODÈLE")
print("=" * 150)

model   = joblib.load("model.pkl") #  Modèle ML entraîné
metrics = joblib.load("metriques.pkl") # les indicateurs de performance

print(f"  Modèle chargé     ✓")
print(f"  MAE  du modèle    : {metrics['mae']}")
print(f"  RMSE du modèle    : {metrics['rmse']}")
print(f"  R²   du modèle    : {metrics['r2']} ({metrics['r2']*100:.1f}%)")

  3.1  CHARGEMENT DU MODÈLE
  Modèle chargé     ✓
  MAE  du modèle    : 3.9413829591183736
  RMSE du modèle    : 4.878659364886242
  R²   du modèle    : 0.506385121941073 (50.6%)


3.2  Fonction de prédiction ─────────────────────────────────────────────

In [22]:
def predire_ventes(stock: float, purchase: float,
                   annee: int, mois: int, jour: int, jour_semaine: int) -> float:

    donnees = np.array([[stock, purchase, annee, mois, jour, jour_semaine]])

    return round(float(model.predict(donnees)[0]), 2)

3.3  Fonction de recommandation ─────────────────────────────────────────

In [23]:
SEUIL_CRITIQUE = 0.20   # commander si stock < 20% de la demande prévue
SEUIL_FAIBLE   = 0.50   # alerte si stock < 50% de la demande prévue

def recommandation(stock: float, ventes_prevues: float) -> dict:
    """
    Retourne la décision et la quantité à commander.
    Niveaux :
      - CRITIQUE  : stock < 20% des ventes prévues → commander maintenant
      - FAIBLE    : stock < 50% des ventes prévues → surveiller
      - SUFFISANT : stock >= ventes prévues        → rien à faire
    """
    ratio = stock / ventes_prevues if ventes_prevues > 0 else 1.0
    quantite = max(0, round(ventes_prevues - stock))

    if ratio < SEUIL_CRITIQUE:
        niveau  = "CRITIQUE"
        message = f"Commander {quantite} unités IMMÉDIATEMENT"
    elif ratio < SEUIL_FAIBLE:
        niveau  = "FAIBLE"
        message = f"Passer commande de {quantite} unités bientôt"
    elif stock < ventes_prevues:
        niveau  = "ATTENTION"
        message = f"Stock légèrement insuffisant ({quantite} unités manquantes)"
    else:
        niveau  = "SUFFISANT"
        message = "Stock suffisant, aucune action requise"

    return {
        "ventes_prevues":    ventes_prevues,
        "stock_actuel":      stock,
        "ratio_stock":       round(ratio * 100, 1),
        "quantite_a_commander": quantite,
        "niveau":            niveau,
        "message":           message,
    }

3.4  Tests sur plusieurs scénarios ──────────────────────────────────────

In [24]:
print("\n" + "=" * 150)
print("  3.2  SCÉNARIOS DE TEST")
print("=" * 150)

scenarios = [
    {"nom": "Stock critique",   "stock": 5,   "purchase": 10, "date": (2026, 6, 18, 3)},
    {"nom": "Stock faible",     "stock": 20,  "purchase": 30, "date": (2026, 6, 19, 4)},
    {"nom": "Stock suffisant",  "stock": 80,  "purchase": 50, "date": (2026, 6, 20, 5)},
    {"nom": "Stock abondant",   "stock": 200, "purchase": 40, "date": (2026, 6, 21, 6)},
]
for s in scenarios:
    annee, mois, jour, jour_semaine = s["date"]

    ventes = predire_ventes(
        s["stock"],
        s["purchase"],
        annee, mois, jour, jour_semaine
    )

    result = recommandation(s["stock"], ventes)

    print(f"\n  Scénario : {s['nom']}")
    print(f"    Stock actuel      : {s['stock']}")
    print(f"    Achats            : {s['purchase']}")
    print(f"    Ventes prévues    : {ventes}")
    print(f"    Ratio stock       : {result['ratio_stock']}%")
    print(f"    Niveau            : {result['niveau']}")
    print(f"    ► {result['message']}")


  3.2  SCÉNARIOS DE TEST

  Scénario : Stock critique
    Stock actuel      : 5
    Achats            : 10
    Ventes prévues    : 26.68
    Ratio stock       : 18.7%
    Niveau            : CRITIQUE
    ► Commander 22 unités IMMÉDIATEMENT


c:\Users\disfra\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\disfra\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(



  Scénario : Stock faible
    Stock actuel      : 20
    Achats            : 30
    Ventes prévues    : 21.15
    Ratio stock       : 94.6%
    Niveau            : ATTENTION
    ► Stock légèrement insuffisant (1 unités manquantes)

  Scénario : Stock suffisant
    Stock actuel      : 80
    Achats            : 50
    Ventes prévues    : 16.65
    Ratio stock       : 480.5%
    Niveau            : SUFFISANT
    ► Stock suffisant, aucune action requise


c:\Users\disfra\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Users\disfra\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(



  Scénario : Stock abondant
    Stock actuel      : 200
    Achats            : 40
    Ventes prévues    : 13.82
    Ratio stock       : 1447.2%
    Niveau            : SUFFISANT
    ► Stock suffisant, aucune action requise


3.5  Prédiction manuelle ────────────────────────────────────────────────

In [25]:
print("\n" + "=" * 55)
print("  3.3  TEST MANUEL (valeurs personnalisées)")
print("=" * 55)

stock_saisi    = 40
purchase_saisi = 25

annee = 2026
mois = 6
jour = 18
jour_semaine = 3

ventes = predire_ventes(
    stock_saisi,
    purchase_saisi,
    annee, mois, jour, jour_semaine
)
result = recommandation(stock_saisi, ventes)

print(f"  Stock    : {stock_saisi}")
print(f"  Achats   : {purchase_saisi}")
print(f"  Ventes prévues : {ventes}")
print(f"  Décision       : {result['niveau']}")
print(f"  ► {result['message']}")



  3.3  TEST MANUEL (valeurs personnalisées)
  Stock    : 40
  Achats   : 25
  Ventes prévues : 16.76
  Décision       : SUFFISANT
  ► Stock suffisant, aucune action requise


c:\Users\disfra\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Objectif : utiliser le modèle pour faire des prédictions,
           calculer la quantité à commander, tester
           plusieurs scénarios.